# Water-cooling estimate

Run both code cells. Starting assumptions are filled in and can be edited. Results show required **US gallons per minute** for water warming by **5°F to 40°F**, every 5°F.

We use the largest copper-loss case from the earlier resistance comparison: **wye wiring, listed resistance treated as per winding**, with RMS current measured in one motor lead.

$$P_{copper}=3I_{RMS}^2R_{listed}\,k_{hot}$$
$$P_{motor,water}=N(P_{copper}+P_{other})f_{water}$$
$$\dot V=\frac{P_{motor,water}+P_{ESC,water}}{\rho c_p\Delta T}$$

This is a resistance-case estimate, not a verified motor rating or a worst case for all heat losses. For wye, treating the resistance as terminal-to-terminal would halve the copper loss. Delta wiring has different conversions. Use **motor-lead RMS amps, not battery amps or the catalog's maximum amps**. Motor count assumes identical operating conditions.

Unverified inputs below are explicitly assumed. Setting other losses to 0 gives a copper-only estimate. A hot-resistance multiplier of 1 means no temperature correction. The water heat fraction is between 0 and 1; choosing 1 assigns all motor losses to water.

## ESC heat

Two [TFL / SEAKING 200A HV V4 ESCs (5–14S)](https://tflnorthamerica.com/product/tfl-hobby-wing-200amp-watercooled-esc/). The listing does not supply efficiency or heat losses; **200 A is a rating, not the current we assume**.

$$P_{ESC,water}=N_{ESC}\,V_{DC}I_{DC}(1-\eta_{ESC})f_{ESC,water}$$

DC current is **per ESC**. Efficiency is a decimal; 0.98 means 98%. The baseline assumes 98%, with 95–99% explored below. Efficiency must cover the ESC's total losses at that condition, including auxiliary loads if applicable. Do not include ESC losses again under motor losses.

The combined result treats all motors and ESCs as loads on the same water circuit with the stated overall temperature rise. The motor and ESC columns show their contributions to that combined flow; separate branches need their own flow allocation.

## Starting values and uncertainty

Your selected motor is **SSS 56154-V2 10Y**; its catalog resistance is **0.0371 Ω**, with measurement basis unconfirmed. Motor and ESC counts are both **2**. All heat is assigned to the water as a conservative cooling-load assumption. Density **1000 kg/m³** is retained; specific heat **4186 J/(kg·K)** is an approximate freshwater assumption, not a confirmed coolant property. **48 V** is an assumed operating supply, consistent with the earlier boat model's roughly 43–48 V design range, not a confirmed battery measurement.

| Uncertain input | Low heat case | Baseline | High heat case |
|---|---:|---:|---:|
| Motor-lead RMS current, A per motor | 50 | 100 | 150 |
| Hot/listed resistance multiplier | 1.0 | 1.25 | 1.5 |
| Other motor losses, W per motor | 0 | 100 | 300 |
| ESC DC input current, A per ESC | 50 | 100 | 150 |
| ESC efficiency | 99% | 98% | 95% |

These are deliberately broad **engineering trial values**, not manufacturer data or measured operating points. Motor current and ESC DC current are independent inputs; equal baseline numbers do not imply they are physically equal. Other losses are placeholders because RPM-dependent data are unavailable. The low case excludes those losses and is not a guaranteed lower bound. High heat combines all higher-loss choices and is not a guaranteed worst case. Resistance is treated as per winding in a wye motor in all cases. Voltage and coolant properties are held fixed.

The calculation flags cases whose estimated motor heat exceeds the electrical power supplied by the ESC. Passing that check does not establish RPM, torque, or a feasible motor operating point. Replace trial currents and efficiencies with telemetry or measured loss data when available.


In [8]:
# Baseline: catalog resistance plus explicit engineering assumptions.
motor = "SSS 56154-V2 10Y"
listed_resistance_ohm = 0.0371       # Catalog value; per-winding interpretation assumed.
motor_lead_current_rms_A = 100      # Assumed motor-lead RMS current, per motor.
hot_resistance_multiplier = 1.25   # Assumed hot/listed resistance ratio.
motor_count = 2
other_losses_per_motor_W = 100      # Assumed allowance, not measured.
heat_fraction_to_water = 1.0        # Assign all motor loss to water.

esc_count = 2
esc_input_voltage_V = 48            # Assumed operating voltage.
esc_dc_current_A = 100              # Assumed DC input current, per ESC.
esc_efficiency = 0.98               # Assumed 98% total efficiency.
esc_heat_fraction_to_water = 1.0     # Assign all ESC loss to water.

water_density_kg_m3 = 1000          # Retained approximate freshwater density.
water_cp_J_kgK = 4186               # Assumed approximate freshwater specific heat.
pump_reference_us_gpm = 1.2         # Your reference; installed flow remains unknown.

# Joint sensitivity cases; currents are trial operating values, not ratings.
#              RMS A, hot R factor, other motor W, ESC DC A, ESC efficiency
low_heat =      (50,   1.0,          0,             50,       0.99)
high_heat =     (150,  1.5,          300,           150,      0.95)


In [9]:
import math


def positive(name, value, zero_ok=False):
    if isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value):
        raise ValueError(f"{name} must be a finite number.")
    if value < 0 or (value == 0 and not zero_ok):
        raise ValueError(f"{name} must be {'nonnegative' if zero_ok else 'positive'}.")


for name, value in {"resistance": listed_resistance_ohm, "density": water_density_kg_m3,
                    "specific heat": water_cp_J_kgK, "voltage": esc_input_voltage_V,
                    "motor count": motor_count, "ESC count": esc_count}.items():
    positive(name, value)
if motor_count != int(motor_count) or esc_count != int(esc_count):
    raise ValueError("Motor and ESC counts must be integers.")
for fraction in (heat_fraction_to_water, esc_heat_fraction_to_water):
    positive("heat fraction", fraction, zero_ok=True)
    if fraction > 1:
        raise ValueError("Heat fractions must be between 0 and 1.")


def heat_case(rms_A, hot_factor, other_W, dc_A, efficiency):
    for name, value in {"RMS current": rms_A, "other losses": other_W, "DC current": dc_A}.items():
        positive(name, value, zero_ok=True)
    positive("hot resistance factor", hot_factor)
    positive("ESC efficiency", efficiency)
    if hot_factor < 1 or efficiency > 1:
        raise ValueError("Hot resistance factor must be >= 1; efficiency must be <= 1.")
    copper_W = 3 * rms_A**2 * listed_resistance_ohm * hot_factor
    motor_loss_W = motor_count * (copper_W + other_W)
    esc_loss_W = esc_count * esc_input_voltage_V * dc_A * (1 - efficiency)
    motor_water_W = motor_loss_W * heat_fraction_to_water
    esc_water_W = esc_loss_W * esc_heat_fraction_to_water
    input_to_motors_W = esc_count * esc_input_voltage_V * dc_A * efficiency
    return {"motor_W": motor_water_W, "esc_W": esc_water_W,
            "total_W": motor_water_W + esc_water_W,
            "energy_check": motor_loss_W <= input_to_motors_W}


cases = {
    "Low": heat_case(*low_heat),
    "Baseline": heat_case(motor_lead_current_rms_A, hot_resistance_multiplier,
                          other_losses_per_motor_W, esc_dc_current_A, esc_efficiency),
    "High": heat_case(*high_heat),
}
print(f"{motor}; {motor_count} motors and {esc_count} ESCs")
print(f"{'Case':10} {'Motor heat [W]':>15} {'ESC heat [W]':>14} {'Total [W]':>12}")
for name, case in cases.items():
    print(f"{name:10} {case['motor_W']:15.1f} {case['esc_W']:14.1f} {case['total_W']:12.1f}")
    if not case['energy_check']:
        print("  INCONSISTENT: motor losses exceed ESC output power; revise these inputs.")

print("\nRequired flow [US gpm]; rise is across the full shared circuit")
print(f"{'Rise [°F]':>10} {'Rise [°C]':>10} {'Low':>10} {'Baseline':>10} {'High':>10}")
sensitivity = []
for rise_F in range(5, 41, 5):
    rise_C = rise_F * 5 / 9
    gpm_per_W = 60000 / (3.785411784 * water_density_kg_m3 * water_cp_J_kgK * rise_C)
    flows = {name: case['total_W'] * gpm_per_W if case['energy_check'] else None
             for name, case in cases.items()}
    sensitivity.append({"rise_F": rise_F, "rise_C": rise_C, **flows})
    values = [f"{flows[name]:10.3f}" if flows[name] is not None else f"{'invalid':>10}"
              for name in cases]
    print(f"{rise_F:10.0f} {rise_C:10.2f} " + " ".join(values))
print(f"\nPump reference: {pump_reference_us_gpm:g} US gpm; actual circuit flow is unverified.")
print("Trial cases, not safe current ratings or verified cooling limits.")


SSS 56154-V2 10Y; 2 motors and 2 ESCs
Case        Motor heat [W]   ESC heat [W]    Total [W]
Low                  556.5           48.0        604.5
Baseline            2982.5          192.0       3174.5
High                8112.8          720.0       8832.8

Required flow [US gpm]; rise is across the full shared circuit
 Rise [°F]  Rise [°C]        Low   Baseline       High
         5       2.78      0.824      4.327     12.040
        10       5.56      0.412      2.164      6.020
        15       8.33      0.275      1.442      4.013
        20      11.11      0.206      1.082      3.010
        25      13.89      0.165      0.865      2.408
        30      16.67      0.137      0.721      2.007
        35      19.44      0.118      0.618      1.720
        40      22.22      0.103      0.541      1.505

Pump reference: 1.2 US gpm; actual circuit flow is unverified.
Trial cases, not safe current ratings or verified cooling limits.


The temperature rises are sensitivity cases, not confirmed allowable temperatures. Water properties are held fixed. This energy balance does not predict winding temperature, jacket heat transfer, or installed pump flow. No design margin is applied.

Sources: [SSS motor specifications](https://tflnorthamerica.com/wp-content/uploads/2020/07/SSS-Specs-2.pdf) · [Wye/delta explanation, p. 13](https://www.maxongroup.com/medias/sys_master/8803451338782.pdf?attachment=true) · [Pump brochure](https://www.mygenerator.com.au/assets/images/resource/pdf/SEAFLO-21_SERIES_40psi_3.8LPM_12V_Water_Pump_Product_Brochure.pdf). Your 1.2 US gpm reference is preserved; the linked 12 V/40 psi pump brochure lists 1.0 GPM / 3.8 LPM.

[Hobbywing ESC specifications](https://www.hobbywing.com/en/products/seaking200ahvv4) identify the 200A HV V4; efficiency assumptions above are not manufacturer specifications.
